In [ ]:
import glicko2
from scipy.stats import poisson, norm
import numpy as np

# --- 1. Setup and Environment Configuration ---

# Glicko-2 Environment: Similar to TrueSkill, it manages the parameters.
# tau is the system constant that constrains the volatility of skill (similar to TrueSkill's tau).
env = glicko2.Borg(tau=0.5) 

# Initial Glicko-2 Ratings: 
# (rating, rating_deviation (RD), volatility)
# rating ~ mu (mean skill)
# RD ~ sigma (uncertainty)
INITIAL_RATING = 1500  # Default Glicko-2 rating
INITIAL_RD = 350       # Default Glicko-2 uncertainty
INITIAL_VOL = 0.06     # Default volatility

# Teams are represented by Glicko2 objects
team_a = env.create_rating(INITIAL_RATING, INITIAL_RD, INITIAL_VOL)
team_b = env.create_rating(INITIAL_RATING, INITIAL_RD, INITIAL_VOL)

# --- 2. Key Parameter Mapping (Poisson Model) ---

# We need a factor to link the difference in Glicko ratings to the expected goal difference.
# This factor is determined through empirical fitting to real-world data.
RATING_TO_GOAL_FACTOR = 0.005 # Example: 200 rating points = 1.0 expected goal difference (200 * 0.005)

def calculate_expected_goals(rating_a, rating_b, factor):
    """
    Calculates the expected goal rate (lambda) for each team based on their ratings.
    
    This is the core of the skill-based outcome model. We assume the *ratio* of 
    expected goals is tied to the rating difference.
    """
    # The rating scale is roughly logarithmic (exponential relationship in the mean)
    rating_diff = rating_a.mu - rating_b.mu 
    
    # Simple model: expected goal rate (lambda) for Team A
    # The term '0.7' is a placeholder for the average goal rate in the league.
    # The factor scales the rating difference into an exponential term for the Poisson rate.
    
    lambda_a = 0.7 * np.exp(factor * rating_diff)
    lambda_b = 0.7 * np.exp(factor * -rating_diff)
    
    # In a typical model, the average expected goals per game (e.g., 1.4 goals) is split between 
    # the teams based on the rating difference.
    
    return lambda_a, lambda_b

def update_ratings_with_score(team_a, team_b, score_a, score_b):
    """
    Updates Glicko-2 ratings based on the actual score, using a calculated 'win probability' 
    derived from the Poisson model.
    """
    lambda_a, lambda_b = calculate_expected_goals(team_a, team_b, RATING_TO_GOAL_FACTOR)

    # --- 3. Calculate Likelihood/Evidence from Scores (Poisson) ---
    
    # Likelihood is P(actual scores | skill parameters)
    # The match outcome is a highly improbable event in the Poisson distribution.
    # We use this likelihood to determine the effective 'weight' of the result.
    
    likelihood_a = poisson.pmf(score_a, lambda_a)
    likelihood_b = poisson.pmf(score_b, lambda_b)
    
    # The ratio of likelihoods can be complex to integrate directly into Glicko-2.
    # Instead, we simplify by determining the 'effective' outcome rank:

    if score_a > score_b:
        # Team A won. The amount of update depends on *how unexpected* the goal margin was.
        # Simple Glicko-2 only needs the outcome rank (1=Win, 0=Loss) and the opponent's rating/RD.
        new_rating_a = env.rate_1vs1(team_a, team_b, outcome=1)
        new_rating_b = env.rate_1vs1(team_b, team_a, outcome=0)
        
    elif score_b > score_a:
        # Team B won.
        new_rating_a = env.rate_1vs1(team_a, team_b, outcome=0)
        new_rating_b = env.rate_1vs1(team_b, team_a, outcome=1)

    else: # Draw
        # Draw is treated as a 0.5 outcome in Glicko-2
        new_rating_a = env.rate_1vs1(team_a, team_b, outcome=0.5)
        new_rating_b = env.rate_1vs1(team_b, team_a, outcome=0.5)
        
    # --- NOTE: The True Statistical Update ---
    # A fully integrated Poisson/Glicko model would iterate the Glicko update 
    # multiple times for blowout scores, effectively weighting the result.
    # Since Glicko-2 is designed for binary outcomes, we stick to the Win/Loss/Draw, 
    # but the rating calculation *itself* already reflects the skill difference 
    # that informed the Poisson lambdas.
    
    # For a simple, illustrative model, we use a heuristic: scale the result.
    # The code below uses a heuristic not officially part of Glicko, but common in practice:
    goal_diff = abs(score_a - score_b)
    
    # If the score difference is high (>= 3), treat it as a more confident result 
    # by artificially lowering the opponent's RD (uncertainty) for the update.
    if goal_diff >= 3 and score_a > score_b:
        team_b_temp = env.create_rating(team_b.mu, team_b.phi * 0.8, team_b.sigma)
        new_rating_a = env.rate_1vs1(team_a, team_b_temp, outcome=1)
        
    return new_rating_a, new_rating_b


# --- 4. Simulation ---

print("--- Initial Ratings ---")
print(f"Team A: Rating={team_a.mu:.2f}, RD={team_a.phi:.2f}")
print(f"Team B: Rating={team_b.mu:.2f}, RD={team_b.phi:.2f}")

# Match 1: Teams are evenly matched, but A wins big (highly unexpected)
SCORE_A_1, SCORE_B_1 = 4, 0
print(f"\n--- Match 1: Team A vs Team B (Result: {SCORE_A_1}-{SCORE_B_1}) ---")

# Update ratings
team_a_new, team_b_new = update_ratings_with_score(team_a, team_b, SCORE_A_1, SCORE_B_1)
team_a, team_b = team_a_new, team_b_new

# Check expected goals based on post-match ratings
lambda_a, lambda_b = calculate_expected_goals(team_a, team_b, RATING_TO_GOAL_FACTOR)

print(f"Team A Rating: {team_a.mu:.2f}, RD: {team_a.phi:.2f}")
print(f"Team B Rating: {team_b.mu:.2f}, RD: {team_b.phi:.2f}")
print(f"Expected Goals (based on new ratings): A={lambda_a:.2f}, B={lambda_b:.2f}")

# Match 2: Team B beats Team A (less unexpected now that the ratings have diverged)
SCORE_A_2, SCORE_B_2 = 1, 2
print(f"\n--- Match 2: Team A vs Team B (Result: {SCORE_A_2}-{SCORE_B_2}) ---")

# Update ratings
team_a_new, team_b_new = update_ratings_with_score(team_a, team_b, SCORE_A_2, SCORE_B_2)
team_a, team_b = team_a_new, team_b_new

lambda_a, lambda_b = calculate_expected_goals(team_a, team_b, RATING_TO_GOAL_FACTOR)

print(f"Team A Rating: {team_a.mu:.2f}, RD: {team_a.phi:.2f}")
print(f"Team B Rating: {team_b.mu:.2f}, RD: {team_b.phi:.2f}")
print(f"Expected Goals (based on new ratings): A={lambda_a:.2f}, B={lambda_b:.2f}")